In [2]:
import os
import numpy as np
import mne
import h5py
import re

In [3]:
def parse_chbmit_summary(summary_path):
    seizures_dict = {}

    with open(summary_path, "r", encoding="latin1") as f:
        text = f.read()

    # split into file sections
    blocks = text.split("File Name:")

    for block in blocks:
        if ".edf" not in block:
            continue

        # ---------------------------------------------------
        # FIX 1: flexible EDF filename matching
        # handles:
        # chb02_16.edf
        # chb02_16+.edf
        # chb02_16_1.edf (future-proof)
        # ---------------------------------------------------
        file_match = re.search(r"(chb\d+_\d+\+?\.edf)", block)

        # fallback: ultra-flexible (safest option)
        if not file_match:
            file_match = re.search(r"(chb\d+_\d+[^\s]*\.edf)", block)

        if not file_match:
            continue

        file_name = file_match.group(1)

        seizures = []

        # ---------------------------------------------------
        # CASE 1: numbered seizures
        # ---------------------------------------------------
        starts_num = re.findall(r"Seizure\s*\d+\s*Start Time:\s*(\d+)", block)
        ends_num   = re.findall(r"Seizure\s*\d+\s*End Time:\s*(\d+)", block)

        if len(starts_num) > 0 and len(ends_num) > 0:
            seizures = [(int(s), int(e)) for s, e in zip(starts_num, ends_num)]

        else:
            # ---------------------------------------------------
            # CASE 2: unnumbered format
            # ---------------------------------------------------
            start = re.search(r"Seizure Start Time:\s*(\d+)", block)
            end   = re.search(r"Seizure End Time:\s*(\d+)", block)

            if start and end:
                seizures = [(int(start.group(1)), int(end.group(1)))]

        seizures_dict[file_name] = seizures

    return seizures_dict





def convert_dataset(seizure_map):
    for file_name, seizures_sec in seizure_map.items():

        edf_path = os.path.join(DATASET_PATH, file_name)
        if not os.path.exists(edf_path):
            print(f"Missing: {edf_path}")
            continue

        print(f"Processing {file_name}")

        # -------------------------
        # Load EDF
        # -------------------------
        raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
        raw.rename_channels(lambda x: x.strip())

        data = raw.get_data().astype(np.float32)   # (channels, samples)
        fs = float(raw.info["sfreq"])
        ch_names = np.array(raw.ch_names, dtype="S")

        # -------------------------
        # Convert seizures (sec → samples)
        # -------------------------
        seizures_samples = np.array(
            [(int(s * fs), int(e * fs)) for s, e in seizures_sec],
            dtype=np.int64
        )

        if seizures_samples.size == 0:
            seizures_samples = np.zeros((0, 2), dtype=np.int64)

        # -------------------------
        # Save HDF5
        # -------------------------
        h5_path = os.path.join(
            OUTPUT_PATH,
            file_name.replace(".edf", ".h5")
        )

        with h5py.File(h5_path, "w") as f:
            f.create_dataset("signals", data=data, compression="gzip")
            f.create_dataset("sampling_rate", data=fs)
            f.create_dataset("channel_names", data=ch_names)
            f.create_dataset("seizure_intervals", data=seizures_samples)

        print(f"Saved → {h5_path} | seizures: {len(seizures_samples)}")



In [5]:
nums = [f"{i:02d}" for i in range(1, 25)]

In [ ]:

for numero in nums:
    texto = f"../Dataset/chb{numero}/chb{numero}-summary.txt"
    seizure_map = parse_chbmit_summary(texto)
    print(seizure_map)



    DATASET_PATH = f"../Dataset/chb{numero}"
    OUTPUT_PATH = f"../Dataset_hdf5/chb{numero}"

    os.makedirs(OUTPUT_PATH, exist_ok=True)



    # RUN
    convert_dataset(seizure_map)



{'chb01_01.edf': [], 'chb01_02.edf': [], 'chb01_03.edf': [(2996, 3036)], 'chb01_04.edf': [(1467, 1494)], 'chb01_05.edf': [], 'chb01_06.edf': [], 'chb01_07.edf': [], 'chb01_08.edf': [], 'chb01_09.edf': [], 'chb01_10.edf': [], 'chb01_11.edf': [], 'chb01_12.edf': [], 'chb01_13.edf': [], 'chb01_14.edf': [], 'chb01_15.edf': [(1732, 1772)], 'chb01_16.edf': [(1015, 1066)], 'chb01_17.edf': [], 'chb01_18.edf': [(1720, 1810)], 'chb01_19.edf': [], 'chb01_20.edf': [], 'chb01_21.edf': [(327, 420)], 'chb01_22.edf': [], 'chb01_23.edf': [], 'chb01_24.edf': [], 'chb01_25.edf': [], 'chb01_26.edf': [(1862, 1963)], 'chb01_27.edf': [], 'chb01_29.edf': [], 'chb01_30.edf': [], 'chb01_31.edf': [], 'chb01_32.edf': [], 'chb01_33.edf': [], 'chb01_34.edf': [], 'chb01_36.edf': [], 'chb01_37.edf': [], 'chb01_38.edf': [], 'chb01_39.edf': [], 'chb01_40.edf': [], 'chb01_41.edf': [], 'chb01_42.edf': [], 'chb01_43.edf': [], 'chb01_46.edf': []}
Processing chb01_01.edf


C:\Users\ritaw\AppData\Local\Temp\ipykernel_12144\456356903.py:74: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)


Saved → ../Dataset_hdf5/chb01\chb01_01.h5 | seizures: 0
Processing chb01_02.edf


C:\Users\ritaw\AppData\Local\Temp\ipykernel_12144\456356903.py:74: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)


Saved → ../Dataset_hdf5/chb01\chb01_02.h5 | seizures: 0
Processing chb01_03.edf


C:\Users\ritaw\AppData\Local\Temp\ipykernel_12144\456356903.py:74: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)


Saved → ../Dataset_hdf5/chb01\chb01_03.h5 | seizures: 1
Processing chb01_04.edf


C:\Users\ritaw\AppData\Local\Temp\ipykernel_12144\456356903.py:74: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)


Saved → ../Dataset_hdf5/chb01\chb01_04.h5 | seizures: 1
Processing chb01_05.edf


C:\Users\ritaw\AppData\Local\Temp\ipykernel_12144\456356903.py:74: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
